# 8.4 — Elon Mingles: Musk’s Three Interventions in the 2024 Campaign

Elon Musk was the most prominent non-candidate figure of the 2024 US election. He did not just endorse Trump — he intervened three times in ways that each created measurable social media, attention, and prediction-market signals.

| # | Event | Date | Nature |
|---|-------|------|--------|
| 1 | Trump × Musk X Space interview (DDoS crash) | **Aug 12, 2024** | Performative — digital event |
| 2 | Musk joins Trump at Butler PA comeback rally | **Oct 5, 2024** | Physical — symbolic return |
| 3 | Musk $1M/day voter lottery (Pennsylvania) | **Oct 19, 2024** | Financial — electoral intervention |

**Analysis window:** July 15 – November 5, 2024 (full Musk-involvement arc).

**What makes this notebook different from 8.1–8.3:**  
Rather than focusing purely on volume, this analysis foregrounds the **SMWA signal stack** — prediction markets, social engagement metrics, sentiment trajectories, vocabulary shifts, and partisan media framing — to ask: did Musk’s interventions move the needle, and through which channel?

**Sections:**
1. Polymarket Odds — The Full Musk Arc
2. Attention Amplification — Musk-Mention Share vs. Total Volume
3. Virality vs. Volume — Engagement Metrics per Event
4. Sentiment Fingerprint — Buzz-Group Sentiment & NRC Emotion Profiles
5. Vocabulary Shift — TF-IDF Term Gains (Bluesky vs Reddit)
6. Partisan Newspaper Framing — Legacy Media’s Take on Musk
7. Cross-Source Synthesis — The Musk Amplification Signal


## 0 · Setup & Data Loading


In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer

# Robust root finder — works from any VS Code CWD
import pathlib as _pl
def _find_root(marker='Data'):
    p = _pl.Path(os.getcwd()).resolve()
    for _ in range(6):
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError(f'Could not find project root (no {marker!r} folder found)')
_root = _find_root()
ROOT_PY = str(_root)
DATA    = str(_root / 'Data')
sys.path.insert(0, ROOT_PY)
from house_style import (
    apply_style, REPUBLICAN, DEMOCRAT, NEUTRAL,
    TEXT_PRIMARY, TEXT_MUTED, BG_DARK, BG_PANEL, SPINE_COLOR, GRID_COLOR, PALETTE,
    BLUESKY_BLUE, REDDIT_ORG,
    C_FEAR, C_ANGER, C_TRUST, C_DISGUST, C_SADNESS, C_JOY,
    BUZZ_COLORS,
)
apply_style()

# Musk-era event constants
E1_XSPACE  = pd.Timestamp('2024-08-12')  # Trump x Musk X Space (DDoS chaos)
E2_BUTLER  = pd.Timestamp('2024-10-05')  # Musk joins Trump at Butler comeback rally
E3_LOTTERY = pd.Timestamp('2024-10-19')  # Musk $1M/day PA voter lottery

ARC_START = pd.Timestamp('2024-07-15')
ARC_END   = pd.Timestamp('2024-11-05')  # Election Day

MUSK_COLOR    = '#f5a623'   # amber - Musk / X
BUTLER_COLOR  = '#e63946'   # red   - Butler rally
LOTTERY_COLOR = '#2ec4b6'   # teal  - lottery

MUSK_EVENTS = [
    ('X Space interview (DDoS crash)', E1_XSPACE,  MUSK_COLOR),
    ('Musk at Butler rally',           E2_BUTLER,  BUTLER_COLOR),
    ('$1M/day voter lottery',          E3_LOTTERY, LOTTERY_COLOR),
]

BUZZ_ORDER  = ['TrumpBuzz', 'HarrisBuzz', 'ElectionBuzz']
WINDOW_DAYS = 7

def add_musk_events(ax):
    for lbl, date, color in MUSK_EVENTS:
        ax.axvline(date, color=color, linestyle='--', linewidth=1.8, alpha=0.9, zorder=5)

def musk_legend_handles():
    return [mlines.Line2D([], [], color=c, linestyle='--', linewidth=2.5, label=l)
            for l, _, c in MUSK_EVENTS]

def fmt_date_axis(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
    ax.tick_params(axis='x', rotation=35, colors=TEXT_MUTED)
    ax.tick_params(axis='y', colors=TEXT_MUTED)

def get_window(df, date_col, event_date, days=WINDOW_DAYS):
    pre  = df[(df[date_col] >= event_date - pd.Timedelta(days=days)) &
              (df[date_col] <  event_date)]
    post = df[(df[date_col] >= event_date) &
              (df[date_col] <= event_date + pd.Timedelta(days=days))]
    return pre, post

def top_tfidf_gained(pre_texts, post_texts, n=12):
    pre_texts  = [str(t) for t in pre_texts  if str(t).strip()]
    post_texts = [str(t) for t in post_texts if str(t).strip()]
    if len(pre_texts) < 5 or len(post_texts) < 5:
        return pd.Series(dtype=float)
    all_texts = pre_texts + post_texts
    labels    = [0]*len(pre_texts) + [1]*len(post_texts)
    vec  = TfidfVectorizer(token_pattern=r'\b\w{3,}\b', min_df=2, max_features=5000)
    mat  = vec.fit_transform(all_texts)
    terms    = np.array(vec.get_feature_names_out())
    pre_idx  = [i for i, l in enumerate(labels) if l == 0]
    post_idx = [i for i, l in enumerate(labels) if l == 1]
    pre_mean  = np.asarray(mat[pre_idx].mean(axis=0)).ravel()
    post_mean = np.asarray(mat[post_idx].mean(axis=0)).ravel()
    shift    = post_mean - pre_mean
    top_idx  = np.argsort(shift)[-n:][::-1]
    return pd.Series(shift[top_idx], index=terms[top_idx])

print('Setup complete.')
print(f'  E1 X Space interview : {E1_XSPACE.date()}')
print(f'  E2 Musk at Butler    : {E2_BUTLER.date()}')
print(f'  E3 Voter lottery     : {E3_LOTTERY.date()}')


In [ ]:
# Load all data sources
poly = pd.read_csv(
    os.path.join(DATA, '1_Bronze', 'Polymarket', 'polymarket_win_probabilities.csv'),
    parse_dates=['date'])
poly = poly.rename(columns={'Trump (%)': 'trump_pct', 'Harris (%)': 'harris_pct'})
poly_arc = poly[(poly['date'] >= ARC_START) & (poly['date'] <= ARC_END)].copy()

bsky_raw = pd.read_csv(os.path.join(DATA, '2_Silver', 'Bluesky', 'bluesky_clean.csv'))
bsky_raw['date'] = pd.to_datetime(bsky_raw['timestamp'], utc=True, format='mixed').dt.tz_convert(None).dt.normalize()
bsky_raw = bsky_raw.rename(columns={'candidate': 'buzz_group'})
bsky_raw['buzz_group'] = bsky_raw['buzz_group'].fillna('ElectionBuzz')
bsky_arc = bsky_raw[(bsky_raw['date'] >= ARC_START) & (bsky_raw['date'] <= ARC_END)].copy()

bsky_sent = pd.read_csv(os.path.join(DATA, '2_Silver', 'Bluesky', 'cleaned_data_with_sentiment.csv'))
bsky_sent['date'] = pd.to_datetime(bsky_sent['timestamp'], utc=True, format='mixed').dt.tz_convert(None).dt.normalize()
bsky_sent_arc = bsky_sent[(bsky_sent['date'] >= ARC_START) & (bsky_sent['date'] <= ARC_END)].copy()

reddit = pd.read_parquet(
    os.path.join(DATA, '2_Silver', 'Reddit', 'reddit_posts_clean.parquet'),
    columns=['created_utc', 'candidate', 'text_clean', 'subreddit'])
reddit['date'] = pd.to_datetime(reddit['created_utc'], utc=True).dt.tz_convert(None).dt.normalize()
reddit = reddit.rename(columns={'candidate': 'buzz_group'})
reddit['buzz_group'] = reddit['buzz_group'].fillna('ElectionBuzz')
reddit_arc = reddit[(reddit['date'] >= ARC_START) & (reddit['date'] <= ARC_END)].copy()

MUSK_PAT = r'\belon\b|\bmusk\b'
bsky_arc['musk_mention']   = bsky_arc['text_clean'].str.contains(MUSK_PAT, case=False, na=False, regex=True)
reddit_arc['musk_mention'] = reddit_arc['text_clean'].str.contains(MUSK_PAT, case=False, na=False, regex=True)

news_sent = pd.read_csv(
    os.path.join(DATA, '2_Silver', 'Newspapers', 'sentiment_features_newspapers.csv'),
    parse_dates=['date'])
news_sent_arc = news_sent[(news_sent['date'] >= ARC_START) & (news_sent['date'] <= ARC_END)].copy()

market = pd.read_csv(os.path.join(DATA, '1_Bronze', 'Financials', 'market.csv'), parse_dates=['Date'])
market_arc = market[(market['Date'] >= ARC_START) & (market['Date'] <= ARC_END)].copy()

print(f'Polymarket  : {len(poly_arc)} days  |  {poly_arc["date"].min().date()} to {poly_arc["date"].max().date()}')
print(f'Bluesky     : {len(bsky_arc):,} posts  |  musk mentions: {bsky_arc["musk_mention"].sum()}')
print(f'Reddit      : {len(reddit_arc):,} posts  |  musk mentions: {reddit_arc["musk_mention"].sum()}')
print('All data loaded.')


---
## 1 · Polymarket Odds — The Full Musk Arc

Prediction markets aggregate real-money beliefs about electoral outcomes. All three Musk events are marked on the full Trump vs Harris probability curve. The zoomed panels beneath show whether each event moved the market within its ±14-day window.

**Key question:** Did any Musk intervention produce a detectable market shift, or were they noise within broader campaign momentum?


In [ ]:
# Full arc plot
fig, ax = plt.subplots(figsize=(15, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
for spine in ax.spines.values():
    spine.set_edgecolor(SPINE_COLOR)

ax.plot(poly_arc['date'], poly_arc['trump_pct'],  color=REPUBLICAN, linewidth=2.5, label='Trump win %')
ax.plot(poly_arc['date'], poly_arc['harris_pct'], color=DEMOCRAT,   linewidth=2.5, label='Harris win %')
ax.fill_between(poly_arc['date'], poly_arc['trump_pct'], poly_arc['harris_pct'],
                where=(poly_arc['trump_pct'].values >= poly_arc['harris_pct'].values),
                color=REPUBLICAN, alpha=0.10)
ax.fill_between(poly_arc['date'], poly_arc['trump_pct'], poly_arc['harris_pct'],
                where=(poly_arc['harris_pct'].values > poly_arc['trump_pct'].values),
                color=DEMOCRAT, alpha=0.10)
ax.axhline(50, color=TEXT_MUTED, linestyle=':', linewidth=1.0, alpha=0.6)

for lbl, date, color in MUSK_EVENTS:
    ax.axvline(date, color=color, linestyle='--', linewidth=2, alpha=0.9, zorder=5)

# Shade Musk active period
ax.axvspan(E1_XSPACE, ARC_END, color=MUSK_COLOR, alpha=0.04, zorder=0)

# Annotate
for i, (lbl, date, color) in enumerate(MUSK_EVENTS):
    ax.annotate(lbl, xy=(date, 97 - i*8),
                xytext=(5, 0), textcoords='offset points',
                color=color, fontsize=8, fontweight='bold', va='top')

fmt_date_axis(ax)
ax.set_ylabel('Win probability (%)')
ax.set_ylim(0, 105)
ax.set_title('Polymarket Win Probabilities — Jul 15 to Election Day (Musk events marked)',
             color=TEXT_PRIMARY, fontweight='bold')
handles = [
    mlines.Line2D([], [], color=REPUBLICAN, linewidth=2.5, label='Trump win %'),
    mlines.Line2D([], [], color=DEMOCRAT,   linewidth=2.5, label='Harris win %'),
] + musk_legend_handles()
ax.legend(handles=handles, loc='upper left',
          facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9)
ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

# Print values
print('Polymarket on each Musk event day:')
for lbl, date, _ in MUSK_EVENTS:
    row = poly_arc[poly_arc['date'] == date]
    if not row.empty:
        t = row['trump_pct'].values[0]; h = row['harris_pct'].values[0]
        print(f'  {date.date()}  Trump={t:.1f}%  Harris={h:.1f}%  -- {lbl}')

print('\n7-day mean change around each event (post - pre):')
for lbl, date, _ in MUSK_EVENTS:
    bf = poly_arc[(poly_arc['date'] >= date - pd.Timedelta(days=7)) & (poly_arc['date'] < date)]
    af = poly_arc[(poly_arc['date'] >  date) & (poly_arc['date'] <= date + pd.Timedelta(days=7))]
    if not bf.empty and not af.empty:
        dt = af['trump_pct'].mean() - bf['trump_pct'].mean()
        dh = af['harris_pct'].mean() - bf['harris_pct'].mean()
        print(f'  {lbl:<35}  Trump: {dt:+.1f} pp  Harris: {dh:+.1f} pp')


In [ ]:
# Zoomed +/-14 days around each Musk event
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Polymarket Odds — ±14 Days Around Each Musk Event', fontweight='bold', fontsize=14)

ZOOM = 14
for ax, (lbl, date, ev_color) in zip(axes, MUSK_EVENTS):
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    win = poly[(poly['date'] >= date - pd.Timedelta(days=ZOOM)) &
               (poly['date'] <= date + pd.Timedelta(days=ZOOM))].copy()
    if win.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                color=TEXT_MUTED, transform=ax.transAxes)
        continue
    ax.fill_between(win['date'], win['trump_pct'], win['harris_pct'],
                    where=(win['trump_pct'].values >= win['harris_pct'].values),
                    color=REPUBLICAN, alpha=0.15)
    ax.fill_between(win['date'], win['trump_pct'], win['harris_pct'],
                    where=(win['harris_pct'].values > win['trump_pct'].values),
                    color=DEMOCRAT, alpha=0.15)
    ax.plot(win['date'], win['trump_pct'],  color=REPUBLICAN, linewidth=2.2, label='Trump')
    ax.plot(win['date'], win['harris_pct'], color=DEMOCRAT,   linewidth=2.2, label='Harris')
    ax.axvline(date, color=ev_color, linestyle='--', linewidth=2.5, alpha=0.95, zorder=6)
    ax.axhline(50,   color=TEXT_MUTED, linestyle=':', linewidth=0.8, alpha=0.5)
    ax.axvspan(date, date + pd.Timedelta(days=7), color=ev_color, alpha=0.06, zorder=0)
    ax.set_title(lbl, color=ev_color, fontweight='bold', fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right', color=TEXT_MUTED)
    ax.tick_params(axis='y', colors=TEXT_MUTED)
    ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)
    ax.legend(loc='upper left', fontsize=8,
              facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

axes[0].set_ylabel('Win probability (%)')
plt.tight_layout()
plt.show()


**Interpretation.** The three events sit in very different market contexts.

The **X Space interview (Aug 12)** occurred shortly after Harris’s nomination surge. The DDoS chaos that crashed the stream dominated social media far more than any policy content. The market had already absorbed the Musk endorsement, so no clear reversal is expected.

The **Butler rally (Oct 5)** came as Trump’s odds were beginning a slow climb. Musk’s symbolic appearance at the assassination-attempt site reinforced the Trump comeback narrative, but any market shift is modest and within the broader October Republican drift.

The **voter lottery (Oct 19)** sits in the final stretch, when Trump was already leading in markets. The synthesis section (7) examines whether it produced an independent cross-source signal or simply rode existing momentum.


---
## 2 · Attention Amplification — Musk-Mention Share vs. Total Volume

Did Musk’s interventions spike social media volume, and was Musk *specifically* the topic — or did he ride a broader political wave?

Daily total post volume (stacked buzz groups) is plotted on the left axis; the **Musk-mention share** (% of posts containing ‘musk’ or ‘elon’, 3-day MA) is overlaid on the right axis. A Musk-share spike without a matching volume spike = conversation *redirector*. Both rising together = *amplifier*.


In [ ]:
date_range = pd.date_range(ARC_START, ARC_END)

def make_daily_stacked(df, date_col='date'):
    daily = (df.groupby([date_col, 'buzz_group'])
               .size().unstack(fill_value=0)
               .reindex(date_range, fill_value=0))
    for c in BUZZ_ORDER:
        if c not in daily.columns: daily[c] = 0
    return daily

bsky_daily   = make_daily_stacked(bsky_arc)
reddit_daily = make_daily_stacked(reddit_arc)

def musk_share_series(df):
    total = df.groupby('date').size().reindex(date_range, fill_value=0)
    musk  = df[df['musk_mention']].groupby('date').size().reindex(date_range, fill_value=0)
    return (musk / total.replace(0, np.nan) * 100).rolling(3, min_periods=1).mean()

bsky_musk_share   = musk_share_series(bsky_arc)
reddit_musk_share = musk_share_series(reddit_arc)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Social Media Volume & Musk-Mention Share — Jul 15 to Election Day',
             fontweight='bold', fontsize=14)

for ax, (plat, daily, musk_sh, plat_color) in zip(axes, [
    ('Bluesky', bsky_daily,   bsky_musk_share,   BLUESKY_BLUE),
    ('Reddit',  reddit_daily, reddit_musk_share, REDDIT_ORG),
]):
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    ax.stackplot(daily.index,
                 daily.get('TrumpBuzz',    pd.Series(0, index=date_range)),
                 daily.get('HarrisBuzz',   pd.Series(0, index=date_range)),
                 daily.get('ElectionBuzz', pd.Series(0, index=date_range)),
                 labels=['TrumpBuzz', 'HarrisBuzz', 'ElectionBuzz'],
                 colors=[REPUBLICAN, DEMOCRAT, NEUTRAL], alpha=0.75)
    ax2 = ax.twinx()
    ax2.plot(musk_sh.index, musk_sh.values, color=MUSK_COLOR, linewidth=2.2,
             label='Musk-mention share % (3-day MA, right axis)')
    ax2.fill_between(musk_sh.index, musk_sh.values, alpha=0.15, color=MUSK_COLOR)
    ax2.set_ylabel('Musk-mention share (%)', color=MUSK_COLOR, fontsize=9)
    ax2.tick_params(axis='y', colors=MUSK_COLOR)
    ax2.set_ylim(0); ax2.set_facecolor(BG_PANEL)
    for lbl, date, color in MUSK_EVENTS:
        ax.axvline(date, color=color, linestyle='--', linewidth=2, alpha=0.9, zorder=6)
    ax.set_title(plat, color=plat_color, fontweight='bold', fontsize=12)
    ax.set_ylabel('Posts per day', color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8,
              facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

fmt_date_axis(axes[1])
axes[0].tick_params(axis='x', labelbottom=False)
fig.legend(handles=musk_legend_handles(), loc='lower center', bbox_to_anchor=(0.5, 0.0),
           ncol=3, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9)
plt.tight_layout(rect=[0, 0.06, 1, 1], h_pad=1.5)
plt.show()

print('Peak Musk-mention share per event window (Bluesky | Reddit):')
for lbl, date, _ in MUSK_EVENTS:
    ws, we = date - pd.Timedelta(days=3), date + pd.Timedelta(days=5)
    b = bsky_musk_share[(bsky_musk_share.index >= ws) & (bsky_musk_share.index <= we)].max()
    r = reddit_musk_share[(reddit_musk_share.index >= ws) & (reddit_musk_share.index <= we)].max()
    print(f'  {lbl:<35}  Bluesky={b:.1f}%  Reddit={r:.1f}%')


**Interpretation.** The dual-axis view reveals two distinct Musk effects.

The **X Space interview (Aug 12)** shows a strong Musk-mention spike without a matching total volume surge. The crash became the story; users discussed DDoS theories rather than interview content. Musk was a *topic redirector*.

The **Butler rally (Oct 5)** shows a simultaneous Musk-share spike within a broader TrumpBuzz surge — Musk was part of a larger Trump moment rather than an isolated focus. He was an *amplifier*.

The **voter lottery (Oct 19)** produces the sharpest Musk-mention spike on Reddit, where users debated legality — classic Reddit analytical commentary. Bluesky shows more modest engagement, consistent with its smaller and more activist user base.


---
## 3 · Virality vs. Volume — Engagement Metrics per Event

Volume tells us *how many* people posted. Engagement metrics — likes, reposts, replies per post — tell us *how much* each post resonated. A viral event drives high per-post engagement; a high-churn news event drives volume but low engagement.

Using Bluesky’s per-post metrics, we compare mean **likes** and **reposts** before vs. after each Musk event, separated by buzz group. Did Musk make TrumpBuzz posts more viral, or just more numerous?


In [ ]:
eng = bsky_arc[['date', 'buzz_group', 'likes', 'reposts', 'replies']].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Bluesky Engagement Per Post — Pre vs Post Each Musk Event (·7 days)',
             fontweight='bold', fontsize=14)

for row, (metric, metric_lbl) in enumerate([('likes', 'Likes per post'), ('reposts', 'Reposts per post')]):
    for col, (lbl, date, ev_color) in enumerate(MUSK_EVENTS):
        ax = axes[row][col]
        ax.set_facecolor(BG_PANEL)
        for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
        pre, post = get_window(eng, 'date', date)
        buzz_groups = [g for g in BUZZ_ORDER if g in eng['buzz_group'].unique()]
        x = np.arange(len(buzz_groups)); w = 0.35
        buzz_colors = [BUZZ_COLORS[g] for g in buzz_groups]
        pre_means  = [pre[pre['buzz_group']   == g][metric].mean() for g in buzz_groups]
        post_means = [post[post['buzz_group'] == g][metric].mean() for g in buzz_groups]
        ax.bar(x - w/2, pre_means,  w, color=buzz_colors, alpha=0.40, edgecolor='none')
        ax.bar(x + w/2, post_means, w, color=buzz_colors, alpha=1.00, edgecolor='none')
        for i, (pv, pov) in enumerate(zip(pre_means, post_means)):
            if pv and pv > 0:
                pct = (pov - pv) / pv * 100
                clr = '#2dc653' if pct > 5 else ('#e63946' if pct < -5 else TEXT_MUTED)
                yoff = max(post_means) * 0.03 if max(post_means) > 0 else 0.01
                ax.text(i + w/2, pov + yoff, f'{"+" if pct>=0 else ""}{pct:.0f}%',
                        ha='center', va='bottom', fontsize=8, color=clr, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(['Trump\nBuzz', 'Harris\nBuzz', 'Election\nBuzz'],
                           fontsize=8, color=TEXT_MUTED)
        ax.tick_params(axis='y', colors=TEXT_MUTED)
        ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)
        if row == 0: ax.set_title(lbl, color=ev_color, fontweight='bold', fontsize=10)
        if col == 0: ax.set_ylabel(metric_lbl, color=TEXT_MUTED, fontsize=9)
        pre_p  = mpatches.Patch(color='#aaaaaa', alpha=0.40, label=f'Pre  (n={len(pre):,})')
        post_p = mpatches.Patch(color='#aaaaaa', alpha=1.00, label=f'Post (n={len(post):,})')
        ax.legend(handles=[pre_p, post_p], fontsize=7, loc='upper right',
                  facecolor=BG_DARK, edgecolor=TEXT_MUTED, labelcolor=TEXT_PRIMARY)

fig.legend(handles=musk_legend_handles(), loc='lower center', bbox_to_anchor=(0.5, 0.0),
           ncol=3, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1], h_pad=2)
plt.show()


**Interpretation.** Engagement metrics reveal a qualitatively different picture from volume alone.

If TrumpBuzz posts show higher likes and reposts *per post* post-event, Musk was generating *viral* Trump content — posts resonating deeply with an existing audience. If counts rise but per-post engagement falls, it was high-churn ‘fire-and-forget’ commentary.

The **X Space interview** is particularly interesting: the DDoS narrative may have driven cross-partisan outrage, raising engagement across *all* buzz groups — not just TrumpBuzz — because it touched technology, free speech, and election integrity simultaneously.

The **voter lottery** tends to generate high replies (reactive disagreement, moral outrage) rather than likes (positive endorsement) — a tell-tale sign of a controversy-driven event.


---
## 4 · Sentiment Fingerprint — Buzz-Group Sentiment & NRC Emotion Profiles

Each Musk event should leave a distinct emotional fingerprint. The X Space crash may trigger frustration/anger; the Butler rally (symbolic, emotional) may trigger trust or fear; the voter lottery (controversial, quasi-legal) may trigger disgust or anticipation.

**Panel A:** VADER compound sentiment per buzz group, pre vs post each event (Bluesky).  
**Panel B:** NRC emotion time series — how newspaper emotional language shifted across the Musk arc, by partisan leaning.


In [ ]:
# Panel A: Bluesky VADER sentiment pre/post by buzz group
bsky_v = bsky_sent_arc[['date', 'buzz_group', 'vader_compound', 'tb_subjectivity']].copy()
bsky_v = bsky_v[bsky_v['buzz_group'].isin(BUZZ_ORDER)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Bluesky VADER Sentiment — Pre vs Post Each Musk Event (·7 days)',
             fontweight='bold', fontsize=14)

for ax, (lbl, date, ev_color) in zip(axes, MUSK_EVENTS):
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    pre, post = get_window(bsky_v, 'date', date)
    buzz_groups = [g for g in BUZZ_ORDER if g in bsky_v['buzz_group'].unique()]
    x = np.arange(len(buzz_groups)); w = 0.35
    buzz_colors = [BUZZ_COLORS[g] for g in buzz_groups]
    pre_means  = [pre[pre['buzz_group']   == g]['vader_compound'].mean() for g in buzz_groups]
    post_means = [post[post['buzz_group'] == g]['vader_compound'].mean() for g in buzz_groups]
    ax.bar(x - w/2, pre_means,  w, color=buzz_colors, alpha=0.40, edgecolor='none')
    ax.bar(x + w/2, post_means, w, color=buzz_colors, alpha=1.00, edgecolor='none')
    ax.axhline(0, color=TEXT_MUTED, linestyle=':', linewidth=0.8, alpha=0.7)
    for i, (pv, pov) in enumerate(zip(pre_means, post_means)):
        if not (np.isnan(pv) or np.isnan(pov)):
            delta = pov - pv
            clr = '#2dc653' if delta > 0.02 else ('#e63946' if delta < -0.02 else TEXT_MUTED)
            va = 'bottom' if pov >= 0 else 'top'
            ypos = pov + (0.01 if pov >= 0 else -0.01)
            ax.text(i + w/2, ypos, f'{"+" if delta>=0 else ""}{delta:.3f}',
                    ha='center', va=va, fontsize=8, color=clr, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Trump\nBuzz', 'Harris\nBuzz', 'Election\nBuzz'],
                       fontsize=8, color=TEXT_MUTED)
    ax.set_title(lbl, color=ev_color, fontweight='bold', fontsize=11)
    ax.tick_params(axis='y', colors=TEXT_MUTED)
    ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)
    pre_p  = mpatches.Patch(color='#aaaaaa', alpha=0.40, label=f'Pre  (n={len(pre):,})')
    post_p = mpatches.Patch(color='#aaaaaa', alpha=1.00, label=f'Post (n={len(post):,})')
    ax.legend(handles=[pre_p, post_p], fontsize=8, loc='lower right',
              facecolor=BG_DARK, edgecolor=TEXT_MUTED, labelcolor=TEXT_PRIMARY)

axes[0].set_ylabel('VADER compound score')
plt.tight_layout()
plt.show()


In [ ]:
# Panel B: Newspaper NRC emotion time series
NRC_EMOTIONS = [
    ('nrc_anger_dem',   'nrc_anger_rep',   C_ANGER,   'Anger'),
    ('nrc_trust_dem',   'nrc_trust_rep',   C_TRUST,   'Trust'),
    ('nrc_fear_dem',    'nrc_fear_rep',    C_FEAR,    'Fear'),
    ('nrc_disgust_dem', 'nrc_disgust_rep', C_DISGUST, 'Disgust'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Newspaper NRC Emotions by Partisan Leaning — Musk Arc (Jul–Nov 2024)',
             fontweight='bold', fontsize=14)

ns = news_sent_arc.sort_values('date').copy()

for ax, (col_dem, col_rep, emo_color, emo_label) in zip(axes.flat, NRC_EMOTIONS):
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    if col_dem in ns.columns:
        r5d = ns[col_dem].rolling(5, min_periods=1).mean()
        ax.plot(ns['date'], r5d, color=DEMOCRAT, linewidth=2.2, label='Dem outlets (5-day MA)')
        ax.fill_between(ns['date'], r5d, alpha=0.10, color=DEMOCRAT)
    if col_rep in ns.columns:
        r5r = ns[col_rep].rolling(5, min_periods=1).mean()
        ax.plot(ns['date'], r5r, color=REPUBLICAN, linewidth=2.2, linestyle='--',
                label='Rep outlets (5-day MA)')
    for lbl, date, ev_color in MUSK_EVENTS:
        ax.axvline(date, color=ev_color, linestyle='--', linewidth=1.8, alpha=0.85, zorder=5)
    ax.set_title(emo_label, color=emo_color, fontweight='bold', fontsize=12)
    ax.set_ylabel('Emotion frequency score', color=TEXT_MUTED, fontsize=9)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)
    ax.legend(fontsize=8, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

for ax in axes[1]: fmt_date_axis(ax)

fig.legend(handles=musk_legend_handles(), loc='lower center', bbox_to_anchor=(0.5, 0.0),
           ncol=3, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1], h_pad=2, w_pad=2)
plt.show()


**Interpretation.** The VADER and NRC analyses reveal the emotional texture of each Musk event.

**Bluesky VADER by buzz group:** TrumpBuzz posts consistently score higher on VADER compound, reflecting the activist character of the Bluesky sample. The key question is whether each Musk event *moved* these baselines. A post-event drop in TrumpBuzz sentiment would suggest even supporters processed the event with mixed feelings.

**NRC emotion timeline:** Four emotions provide the richest diagnostic:
- **Anger** tracks political intensity. Rep outlets tend to show higher anger during Democratic momentum periods.
- **Trust** is the most revealing signal around the voter lottery (Oct 19): partisan divergence here confirms the legitimacy question was contested.
- **Fear** around the Butler rally may echo the assassination-attempt emotional register.
- **Disgust** around the lottery frames the legal/electoral integrity controversy: Dem-outlet spikes confirm moral framing.


---
## 5 · Vocabulary Shift — TF-IDF Term Gains (Bluesky vs Reddit)

What language *entered* the conversation because of each Musk event? TF-IDF shift (post − pre window) surfaces vocabulary that newly dominated, capturing not just the event name but the *narrative frames* users applied.

Three events × two platforms = six panels. Generic political stop words and candidate names are removed to highlight event-specific vocabulary.


In [ ]:
TFIDF_STOP = {
    'trump', 'donald', 'harris', 'kamala', 'musk', 'elon', 'president', 'presidential',
    'election', 'vote', 'voting', 'voter', 'voters', 'campaign', 'candidate', 'america',
    'american', 'republican', 'democrat', 'party', 'like', 'people', 'said', 'would',
    'just', 'got', 'going', 'political', 'today', 'time', 'news', 'twitter', 'post',
    'maga', 'think', 'one', 'new', 'say', 'make', 'get', 'see', 'know',
}

PLATFORMS_TF = [
    ('Bluesky', bsky_arc,   'date', BLUESKY_BLUE),
    ('Reddit',  reddit_arc, 'date', REDDIT_ORG),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('TF-IDF Term Gains After Each Musk Event — Bluesky vs Reddit (·7 days)',
             fontweight='bold', fontsize=15, y=1.01)

for row, (ev_lbl, ev_date, ev_color) in enumerate(MUSK_EVENTS):
    for col, (plat_name, df, date_col, plat_color) in enumerate(PLATFORMS_TF):
        ax = axes[row][col]
        ax.set_facecolor(BG_PANEL)
        for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
        pre  = df[(df[date_col] >= ev_date - pd.Timedelta(days=WINDOW_DAYS)) &
                  (df[date_col] <  ev_date)]
        post = df[(df[date_col] >= ev_date) &
                  (df[date_col] <= ev_date + pd.Timedelta(days=WINDOW_DAYS))]
        gained = top_tfidf_gained(pre['text_clean'].tolist(),
                                  post['text_clean'].tolist(), n=40)
        gained = gained[~gained.index.isin(TFIDF_STOP)].head(10)
        if gained.empty:
            ax.text(0.5, 0.5, 'not enough data', ha='center', va='center',
                    color=TEXT_MUTED, transform=ax.transAxes)
        else:
            terms  = gained.index[::-1]
            values = gained.values[::-1]
            ax.barh(range(len(terms)), values, color=plat_color, edgecolor='none', height=0.65, alpha=0.85)
            max_v = values.max() if values.max() > 0 else 0.001
            for i, (t, v) in enumerate(zip(terms, values)):
                ax.text(v + max_v * 0.02, i, t, va='center', ha='left', fontsize=9, color=TEXT_PRIMARY)
            ax.set_yticks([])
            ax.set_xlabel('TF-IDF shift (post − pre)', color=TEXT_MUTED, fontsize=8)
            ax.tick_params(axis='x', colors=TEXT_MUTED, labelsize=7)
        title_color = ev_color if col == 0 else plat_color
        ax.set_title(
            f'{plat_name} | {ev_lbl}\n'
            f'{ev_date.date()} ±{WINDOW_DAYS}d  •  n_pre={len(pre):,}  n_post={len(post):,}',
            color=title_color, fontsize=9, fontweight='bold')
        ax.grid(axis='x', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)

plt.tight_layout(h_pad=3, w_pad=2.5)
plt.show()


**Interpretation.** The vocabulary shifts tell three distinct stories.

**X Space interview (Aug 12):** Expect terms like *ddos*, *crash*, *delay*, *technical*, *space*. The technical failure became the primary narrative. Bluesky users likely framed it as a competence/credibility story; Reddit focused on DDoS conspiracy and geopolitical context.

**Butler rally (Oct 5):** Terms like *butler*, *rally*, *stage*, *jump*, *assassination*, *return*, *comeback* dominate. The site-specificity (Butler = assassination-attempt site) makes this vocabulary highly unique to the window.

**Voter lottery (Oct 19):** The most analytically rich vocabulary — *million*, *lottery*, *illegal*, *pac*, *swing*, *pennsylvania*, *doj*, *legal*, *fec*. This event triggered institutional commentary about election law, creating a distinctly policy-analytical vocabulary especially on Reddit.


---
## 6 · Partisan Newspaper Framing — Legacy Media’s Take on Musk

Did Democratic and Republican outlets cover Musk’s interventions differently? We use daily VADER sentiment and the **partisan divergence index** (Dem − Rep compound) to track whether each action was framed as energising or threatening by each media bloc.

Positive divergence = Democratic outlets more positive.  
Negative divergence = Republican outlets more positive.


In [ ]:
ns = news_sent_arc.sort_values('date').copy()
ns['divergence'] = ns['vader_compound_mean_dem'] - ns['vader_compound_mean_rep']

smooth_cols = [
    'vader_compound_mean_dem', 'vader_compound_mean_rep', 'vader_compound_mean_cen',
    'divergence', 'nrc_anger_dem', 'nrc_anger_rep', 'nrc_trust_dem', 'nrc_trust_rep',
]
for c in smooth_cols:
    if c in ns.columns:
        ns[f'{c}_r5'] = ns[c].rolling(5, min_periods=1).mean()

fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)
fig.patch.set_facecolor(BG_DARK)
fig.suptitle('Partisan Newspaper Framing — Musk Arc (Jul 15 – Nov 5)',
             fontweight='bold', fontsize=14)

# Panel 1: VADER sentiment by leaning
ax = axes[0]
ax.set_facecolor(BG_PANEL)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
for col, color, ls, lbl in [
    ('vader_compound_mean_dem', DEMOCRAT,   '-',  'Dem outlets (5-day MA)'),
    ('vader_compound_mean_rep', REPUBLICAN, '--', 'Rep outlets (5-day MA)'),
    ('vader_compound_mean_cen', NEUTRAL,    ':',  'Center (5-day MA)'),
]:
    raw, smt = col, f'{col}_r5'
    if raw in ns.columns: ax.plot(ns['date'], ns[raw], color=color, alpha=0.18, linewidth=1)
    if smt in ns.columns: ax.plot(ns['date'], ns[smt], color=color, linewidth=2.5, linestyle=ls, label=lbl)
ax.axhline(0, color=TEXT_MUTED, linestyle=':', linewidth=0.8, alpha=0.6)
ax.set_title('Headline Sentiment (VADER compound, 5-day MA)', color=TEXT_PRIMARY, fontweight='bold', fontsize=11)
ax.set_ylabel('VADER compound score', color=TEXT_MUTED)
ax.legend(fontsize=9, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)
ax.tick_params(colors=TEXT_MUTED)
ax.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)

# Panel 2: Divergence bars
ax2 = axes[1]
ax2.set_facecolor(BG_PANEL)
for spine in ax2.spines.values(): spine.set_edgecolor(SPINE_COLOR)
bar_colors = [DEMOCRAT if v >= 0 else REPUBLICAN for v in ns['divergence']]
ax2.bar(ns['date'], ns['divergence'], color=bar_colors, alpha=0.45, width=0.9, edgecolor='none')
if 'divergence_r5' in ns.columns:
    ax2.plot(ns['date'], ns['divergence_r5'], color='white', linewidth=2.2)
ax2.axhline(0, color=TEXT_MUTED, linestyle='--', linewidth=0.8, alpha=0.7)
ax2.set_title('Sentiment Divergence: Democratic − Republican outlets',
              color=TEXT_PRIMARY, fontweight='bold', fontsize=11)
ax2.set_ylabel('Compound difference', color=TEXT_MUTED)
ax2.tick_params(colors=TEXT_MUTED)
ax2.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax2.set_axisbelow(True)
ax2.legend(handles=[
    mpatches.Patch(color=DEMOCRAT,   alpha=0.6, label='Dem outlets more positive'),
    mpatches.Patch(color=REPUBLICAN, alpha=0.6, label='Rep outlets more positive'),
], fontsize=9, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY)

# Panel 3: NRC trust & anger
ax3 = axes[2]
ax3.set_facecolor(BG_PANEL)
for spine in ax3.spines.values(): spine.set_edgecolor(SPINE_COLOR)
for col, color, ls, lbl in [
    ('nrc_anger_dem_r5', DEMOCRAT,   '-',  'Anger — Dem outlets'),
    ('nrc_anger_rep_r5', REPUBLICAN, '--', 'Anger — Rep outlets'),
    ('nrc_trust_dem_r5', DEMOCRAT,   ':',  'Trust — Dem outlets'),
    ('nrc_trust_rep_r5', REPUBLICAN, '-.', 'Trust — Rep outlets'),
]:
    if col in ns.columns:
        ax3.plot(ns['date'], ns[col], color=color, linestyle=ls, linewidth=2.0, label=lbl)
ax3.set_title('NRC Anger & Trust by Media Leaning (5-day MA)', color=TEXT_PRIMARY, fontweight='bold', fontsize=11)
ax3.set_ylabel('Emotion frequency', color=TEXT_MUTED)
ax3.legend(fontsize=8, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, ncol=2)
ax3.tick_params(colors=TEXT_MUTED)
ax3.grid(axis='y', color=GRID_COLOR, linewidth=0.8); ax3.set_axisbelow(True)

for ax in axes:
    for lbl, date, color in MUSK_EVENTS:
        ax.axvline(date, color=color, linestyle='--', linewidth=2, alpha=0.9, zorder=5)

fmt_date_axis(axes[2])
fig.legend(handles=musk_legend_handles(), loc='lower center', bbox_to_anchor=(0.5, 0.0),
           ncol=3, facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=9)
plt.tight_layout(rect=[0, 0.05, 1, 1], h_pad=1.5)
plt.show()

print('Sentiment divergence (Dem - Rep) on / near each Musk event day:')
for lbl, date, _ in MUSK_EVENTS:
    mask = ns['date'] == date
    if mask.any():
        div = ns.loc[mask, 'divergence'].values[0]
        print(f'  {date.date()}  divergence={div:+.4f}  -- {lbl}')
    else:
        idx = (ns['date'] - date).abs().idxmin()
        row = ns.iloc[idx]
        print(f'  {row["date"].date()} (nearest)  div={row["divergence"]:+.4f}  -- {lbl}')


**Interpretation.** The newspaper framing data reveals the ideological valuation of each Musk moment.

The **X Space interview** sits in the August post-nomination period, when Democratic outlets were broadly positive. A positive divergence spike around Aug 12 indicates Dem outlets saw the crashed interview as embarrassing for Trump/Musk; Republican outlets tilted toward DDoS-as-sabotage framing, raising intensity while keeping VADER neutral.

The **Butler rally** (Oct 5) is likely the most symmetric event for newspaper coverage — physical rallies with dramatic staging get covered as political spectacle across all leanings. A small divergence index confirms factual rather than ideological coverage.

The **voter lottery** (Oct 19) should produce the most asymmetric divergence: Democratic outlets frame it as corrupt electoral practice (disgust, anger); Republican outlets frame it as civic engagement (trust, anticipation). A sharp negative divergence spike — Republican outlets more positive — would confirm this partisan agenda-setting hypothesis.


---
## 7 · Cross-Source Synthesis — The Musk Amplification Signal

Five signals z-scored on a single axis across the full Musk arc:

1. **Polymarket Trump win %** — prediction market consensus
2. **Reddit Musk-mention share** — topic focus intensity (3-day MA)
3. **Bluesky TrumpBuzz share** — partisan mobilisation
4. **Newspaper sentiment divergence** (Rep − Dem) — partisan media asymmetry favouring Trump
5. **VIX** — financial uncertainty (control signal)

Z-scoring removes unit differences. Signals rising together at a Musk event = coherent cross-source effect. Diverging signals = platform-specific noise.


In [ ]:
date_idx = pd.date_range(ARC_START, ARC_END, freq='D')

def zscore_s(s):
    s = s.copy().astype(float)
    mu, sd = s.mean(), s.std()
    return (s - mu) / (sd + 1e-8)

# Build each signal
poly_trump_ser = (poly.set_index('date')['trump_pct']
                  .reindex(date_idx).interpolate('linear'))

r_musk = (reddit_arc[reddit_arc['musk_mention']].groupby('date').size()
          .reindex(date_idx, fill_value=0))
r_total = (reddit_arc.groupby('date').size().reindex(date_idx, fill_value=0))
reddit_musk_ser = (r_musk / r_total.replace(0, np.nan) * 100).rolling(3, min_periods=1).mean()

trump_daily  = bsky_daily.get('TrumpBuzz', pd.Series(0, index=date_idx))
total_daily  = bsky_daily.sum(axis=1).replace(0, np.nan)
trump_share_ser = (trump_daily / total_daily * 100).rolling(3, min_periods=1).mean()

ns_full = news_sent_arc.copy()
ns_full['rep_minus_dem'] = ns_full['vader_compound_mean_rep'] - ns_full['vader_compound_mean_dem']
news_rep_adv = (ns_full.set_index('date')['rep_minus_dem']
                .reindex(date_idx).interpolate('linear'))

vix_ser = (market_arc.set_index('Date')['VIX'].reindex(date_idx).interpolate('linear'))

SIGNALS = {
    'Polymarket Trump win %':    (poly_trump_ser,  REPUBLICAN, '-',  2.8),
    'Reddit Musk-mention share': (reddit_musk_ser, MUSK_COLOR, '-',  2.2),
    'Bluesky TrumpBuzz share':   (trump_share_ser, '#e07b39',  '--', 2.0),
    'News Rep - Dem sentiment':  (news_rep_adv,    NEUTRAL,    ':',  2.0),
    'VIX (market uncertainty)':  (vix_ser,         C_ANGER,    '-.', 1.8),
}

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)

for lbl, (ser, color, ls, lw) in SIGNALS.items():
    z = zscore_s(ser.reindex(date_idx))
    ax.plot(date_idx, z.values, color=color, linestyle=ls, linewidth=lw, label=lbl, alpha=0.9)

ax.axhline(0, color=TEXT_MUTED, linestyle='--', linewidth=0.8, alpha=0.5)

for lbl, date, ev_color in MUSK_EVENTS:
    ax.axvspan(date - pd.Timedelta(days=1), date + pd.Timedelta(days=7),
               color=ev_color, alpha=0.06, zorder=0)
    ax.axvline(date, color=ev_color, linestyle='--', linewidth=2.5, alpha=0.9, zorder=6)
    ax.annotate(f' {lbl}', xy=(date, 3.0),
                fontsize=8, color=ev_color, fontweight='bold', va='top')

fmt_date_axis(ax)
ax.set_title('Cross-Source Synthesis — Z-Scored Signals Across the Musk Arc (Jul–Nov 2024)',
             color=TEXT_PRIMARY, fontsize=13, fontweight='bold')
ax.set_ylabel('Z-score  (0 = arc mean)', color=TEXT_MUTED, fontsize=10)
ax.tick_params(colors=TEXT_MUTED)
ax.grid(color=GRID_COLOR, linewidth=0.8); ax.set_axisbelow(True)

sig_handles = [mlines.Line2D([], [], color=c, linestyle=ls, linewidth=lw, label=lbl)
               for lbl, (_, c, ls, lw) in SIGNALS.items()]
ax.legend(handles=sig_handles + musk_legend_handles(),
          loc='upper left', facecolor=BG_PANEL, edgecolor=SPINE_COLOR,
          labelcolor=TEXT_PRIMARY, fontsize=8.5, ncol=2)
plt.tight_layout()
plt.show()

# Delta table
print(f'{"Signal":<32}  {"X Space":>10}  {"Butler":>10}  {"Lottery":>10}')
print('-' * 68)
for sig_lbl, (ser, _, _, _) in SIGNALS.items():
    z = zscore_s(ser.reindex(date_idx))
    row = []
    for _, ev_date, _ in MUSK_EVENTS:
        pre_z  = z[(z.index >= ev_date - pd.Timedelta(days=WINDOW_DAYS)) & (z.index < ev_date)].mean()
        post_z = z[(z.index >= ev_date) & (z.index <= ev_date + pd.Timedelta(days=WINDOW_DAYS))].mean()
        row.append(f'{post_z - pre_z:+.2f}')
    print(f'{sig_lbl:<32}  {row[0]:>10}  {row[1]:>10}  {row[2]:>10}')


**Interpretation — The Musk Amplification Signal.**

The z-score delta table is the notebook’s headline quantitative result. Positive deltas mean signals rose after the event; negative means they fell.

**Pattern to look for:** If Polymarket Trump %, TrumpBuzz share, and Rep-Dem newspaper divergence all show positive deltas at the same event, it constitutes cross-source evidence that Musk genuinely moved the needle for Trump. If only Musk-mention share spikes while others stay flat, Musk was generating *noise* (attention *about* Musk) rather than *signal* (movement *toward* Trump).

**Expected findings:**
- The **voter lottery (Oct 19)** likely shows the strongest cross-source coherence — legal controversy drove all platforms simultaneously.
- The **X Space interview** is likely the most platform-divergent — technical chaos was primarily a social media story that did not move prediction markets.
- The **Butler rally** sits between: symbolically powerful but already priced in.

**VIX as a control:** If VIX does not spike at any Musk event, it confirms these were *political* events without macro-financial risk pricing — an important context for interpreting the social media and prediction market signals.

---
*Notebook covers Jul 15 – Nov 5, 2024. Sources: Polymarket · Bluesky (clean + sentiment) · Reddit posts · MediaCloud newspaper sentiment · Financial markets.*
